# Phase 4A2 -- TaViT Evaluation: Temporal Trajectory Assessment

> **Purpose**: Evaluate TaViT trajectory embeddings (256-D per patient) against the
> static per-scan embeddings on temporal tests T1-T8, plus new trajectory-specific metrics.
>
> **Inputs**:
> - `tavit_trajectory_embeddings.npz` (from Phase4_A1)
> - `vit_swinunetr_embeddings_v5_hybrid.npz` (static baseline)
> - `tumor_volumes.csv`
>
> **No GPU required** -- pure numpy/sklearn/scipy.


In [ ]:
import numpy as np
import json as _json
import warnings
import os
from pathlib import Path
from collections import defaultdict

import pandas as pd
from scipy.stats import spearmanr, kendalltau, mannwhitneyu
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import RidgeClassifier, Ridge
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (f1_score, roc_auc_score, cohen_kappa_score,
                             classification_report, silhouette_score)
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
np.random.seed(42)

OUTPUT_ROOT = Path("/kaggle/working/tavit_evaluation")
FIG_DIR = OUTPUT_ROOT / "figures"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

def find_file(names):
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for f in root.rglob("*"):
            for name in names:
                if name in f.name:
                    return f
    return None

# ============================================================
# LOAD TAVIT TRAJECTORY EMBEDDINGS
# ============================================================
traj_path = find_file(["tavit_trajectory_embeddings.npz"])
assert traj_path is not None, "tavit_trajectory_embeddings.npz not found!"
traj_npz = np.load(traj_path, allow_pickle=True)
traj_pids = sorted(traj_npz.files)
print(f"TaViT trajectories: {len(traj_pids)} patients | "
      f"dim={traj_npz[traj_pids[0]].shape[0]}")
print(f"  File: {traj_path}")

# Load trajectory metadata if available
meta_path = find_file(["tavit_trajectory_meta.json"])
traj_meta = {}
if meta_path:
    with open(meta_path) as f:
        traj_meta = _json.load(f)
    print(f"  Metadata: {len(traj_meta)} entries")

# ============================================================
# ============================================================
# LOAD STATIC HYBRID EMBEDDINGS (for comparison)
# ============================================================
static_path = find_file(["vit_swinunetr_embeddings_v5_hybrid.npz",
                          "vit_swinunetr_embeddings_v5_noglobal.npz"])
static_npz = None
static_patient_seqs = {}
if static_path:
    raw_npz = np.load(static_path, allow_pickle=True)
    all_npz_keys = sorted(raw_npz.files)
    print(f"\nStatic hybrid NPZ keys ({len(all_npz_keys)}): {all_npz_keys[:4]}")
    print(f"  File: {static_path}")

    # Detect format: per-scan keys vs batch arrays
    def is_scan_key(k):
        return "__" in k and any(k.split("__")[1].startswith(p) for p in ["t","p"])

    format_a_keys = [k for k in all_npz_keys if is_scan_key(k)]
    if len(format_a_keys) >= len(all_npz_keys) * 0.5:
        # Format A: per-scan keys
        print(f"  Format: PER-SCAN keys ({len(format_a_keys)} scans)")
        scan_keys = format_a_keys
        scan_dict = {k: raw_npz[k].astype(np.float32) for k in scan_keys}
    else:
        # Format B: batch arrays (embeddings + patient_ids + timepoints)
        print(f"  Format: BATCH arrays")
        emb_key = next((k for k in all_npz_keys if raw_npz[k].ndim == 2), None)
        pid_key = next((k for k in all_npz_keys if 'patient' in k.lower() or 'pid' in k.lower()), None)
        tp_key  = next((k for k in all_npz_keys if 'time' in k.lower() or k.lower() in ['tp','tps']), None)
        emb_matrix = raw_npz[emb_key].astype(np.float32)
        raw_pids = [str(s) for s in raw_npz[pid_key]] if pid_key else [f"s{i}" for i in range(len(emb_matrix))]
        raw_tps  = list(raw_npz[tp_key]) if tp_key else list(range(len(emb_matrix)))
        scan_keys = [f"{raw_pids[i]}__t{int(raw_tps[i])}" for i in range(len(emb_matrix))]
        scan_dict = {scan_keys[i]: emb_matrix[i] for i in range(len(emb_matrix))}
        print(f"  Composite keys: {len(scan_dict)} | example: {scan_keys[:2]}")

    static_npz = raw_npz  # keep reference
    dim_static = next(iter(scan_dict.values())).shape[0]
    print(f"  Embedding dim: {dim_static} | Total scans: {len(scan_dict)}")

    for key in scan_keys:
        parts = key.split("__")
        if len(parts) != 2:
            continue
        pid, tp_str = parts
        tp = int("".join(filter(str.isdigit, tp_str)) or "0")
        emb = scan_dict[key]
        if pid not in static_patient_seqs:
            static_patient_seqs[pid] = []
        static_patient_seqs[pid].append((tp, emb, key))
    for pid in static_patient_seqs:
        static_patient_seqs[pid].sort(key=lambda x: x[0])
    print(f"  Static patients: {len(static_patient_seqs)}")
else:
    print("\n  WARNING: Static hybrid embeddings not found -- will skip comparison")

# ============================================================
# LOAD TUMOR VOLUMES
# ============================================================
vol_path = find_file(["tumor_volumes.csv"])
vol_df = None
vol_lookup = {}
if vol_path:
    vol_df = pd.read_csv(vol_path)
    print(f"\nTumor volumes: {len(vol_df)} rows from {vol_path.name}")
    print(f"  Columns: {list(vol_df.columns)}")
    cols = list(vol_df.columns)
    pid_col = next((c for c in cols if any(k in c.lower() for k in
                    ['patient', 'pid', 'subject', 'case', 'id'])), None)
    tp_col  = next((c for c in cols if any(k in c.lower() for k in
                    ['time', 'visit', 'tp', 'scan', 'follow'])), None)
    vol_col = next((c for c in cols if any(k in c.lower() for k in
                    ['vol', 'size', 'mm3', 'tumor', 'wt', 'whole', 'total'])), None)
    if vol_col is None:
        num_cols = vol_df.select_dtypes(include=['float64','float32','int64']).columns.tolist()
        vol_col = next((c for c in num_cols if c != tp_col), None)
    print(f"  Detected: pid_col={pid_col} | tp_col={tp_col} | vol_col={vol_col}")
    if pid_col and vol_col:
        for i, row in vol_df.iterrows():
            pid_val = str(row[pid_col])
            tp_val  = int(row[tp_col]) if tp_col and not pd.isna(row[tp_col]) else i
            vol_val = float(row[vol_col]) if not pd.isna(row[vol_col]) else 0.0
            vol_lookup[(pid_val, tp_val)] = vol_val
        print(f"  Volume lookup: {len(vol_lookup)} entries")
else:
    print("\n  WARNING: tumor_volumes.csv not found -- attach brats2024-metadata dataset!")

def fast_vol_lookup(pid, tp):
    tp_int = int(str(tp).replace("t", "").replace("p", ""))
    for key_pid in [pid, pid.split("-")[-1]]:
        if (key_pid, tp_int) in vol_lookup:
            return vol_lookup[(key_pid, tp_int)]
    for (k_pid, k_tp), v in vol_lookup.items():
        if tp_int == k_tp and (pid in k_pid or k_pid in pid):
            return v
    return None

# ============================================================
# BUILD PATIENT-LEVEL LABELS (using traj_meta timepoints)
# ============================================================
# Use traj_meta for timepoints so we don't depend on static_patient_seqs
patient_labels = {}
for pid in traj_pids:
    meta = traj_meta.get(pid, {})
    tps = meta.get("timepoints", [])
    n_scans = meta.get("n_scans", len(tps))
    if len(tps) < 2:
        continue
    first_tp, last_tp = tps[0], tps[-1]
    v_first = fast_vol_lookup(pid, first_tp)
    v_last  = fast_vol_lookup(pid, last_tp)
    if v_first is not None and v_last is not None and v_first > 1.0:
        delta = (v_last - v_first) / v_first
        response = "progressive" if delta > 0.25 else "responder" if delta < -0.25 else "stable"
        patient_labels[pid] = {
            "delta_vol": delta,
            "v_first": v_first,
            "v_last": v_last,
            "response": response,
            "n_scans": n_scans,
        }

print(f"\nPatients with volume labels: {len(patient_labels)}")
if len(patient_labels) == 0:
    print("  vol_lookup empty -- is tumor_volumes.csv attached?")
    print(f"  vol_lookup size: {len(vol_lookup)} | traj_meta size: {len(traj_meta)}")
    if traj_meta:
        sample_pid = list(traj_meta.keys())[0]
        print(f"  traj_meta sample: {sample_pid} -> {traj_meta[sample_pid]}")
else:
    resp_counts = defaultdict(int)
    for v in patient_labels.values():
        resp_counts[v["response"]] += 1
    for r in ["progressive", "stable", "responder"]:
        print(f"  {r}: {resp_counts[r]}")
if patient_labels:
    resp_counts = defaultdict(int)
    for v in patient_labels.values():
        resp_counts[v["response"]] += 1
    for r in ["progressive", "stable", "responder"]:
        print(f"  {r}: {resp_counts[r]}")

# Split info
splits = defaultdict(list)
for pid in traj_pids:
    sp = traj_meta.get(pid, {}).get("split", "unknown")
    splits[sp].append(pid)
for sp in ["train", "val", "test", "unknown"]:
    if splits[sp]:
        print(f"  {sp}: {len(splits[sp])} patients")

print(f"\n{'='*60}")
print(f"  DATA LOADED SUCCESSFULLY")
print(f"{'='*60}")


In [ ]:
# ============================================================
# TEMPORAL TESTS T1-T8: TaViT vs STATIC
# ============================================================
print("=" * 60)
print("  TEMPORAL TESTS: TaViT TRAJECTORY vs STATIC EMBEDDING")
print("=" * 60)

results = {"tavit": {}, "static_hybrid": {}}

# -- Helper: compute temporal metrics for a set of patient embeddings --
def compute_temporal_metrics(patient_embs, patient_seqs_dict, label="model"):
    """
    patient_embs: dict pid -> embedding (for TaViT: 256-D trajectory,
                                         for static: list of per-scan embs)
    Returns dict of temporal metric values.
    """
    metrics = {}

    # Determine embedding type from first element
    first_val = next(iter(patient_embs.values()), None)
    is_trajectory = not isinstance(first_val, list)

    # Collect paired data: for each patient, get (emb_distance, volume_change)
    spearman_pairs = []  # (embedding_dist, volume_change)
    kendall_pairs = []
    ordering_pass = []
    coherence_vals = []
    direction_pairs = []  # (predicted_direction, actual_direction)
    rano_pairs = []
    n_matched = 0

    for pid in sorted(patient_embs.keys()):
        if pid not in patient_seqs_dict or pid not in patient_labels:
            continue

        seqs = patient_seqs_dict[pid]
        if len(seqs) < 2:
            continue

        pl = patient_labels[pid]
        n_matched += 1

        if is_trajectory:
            # TaViT: use trajectory embedding directly
            traj_emb = patient_embs[pid]

            # T1: correlate trajectory embedding norm with absolute volume change
            spearman_pairs.append((np.linalg.norm(traj_emb), abs(pl["delta_vol"])))

            # T8: correlate first PC of trajectory with signed volume change
            kendall_pairs.append((traj_emb, pl["delta_vol"]))

        else:
            # Static: compute pairwise distances between consecutive scans
            emb_list = patient_embs[pid]
            for i in range(len(emb_list) - 1):
                e1, tp1 = emb_list[i]
                e2, tp2 = emb_list[i + 1]
                v1 = fast_vol_lookup(pid, tp1)
                v2 = fast_vol_lookup(pid, tp2)
                if v1 is None or v2 is None or v1 < 1.0:
                    continue

                dist = np.linalg.norm(e1 - e2)
                dv = (v2 - v1) / v1
                spearman_pairs.append((dist, abs(dv)))

                # Direction: did embedding move in consistent direction with volume?
                if abs(dv) > 0.05:
                    direction_pairs.append((dist, 1.0 if dv > 0 else 0.0))

            # Coherence: consecutive embedding distances should be smooth
            if len(emb_list) >= 3:
                dists = []
                for i in range(len(emb_list) - 1):
                    dists.append(np.linalg.norm(emb_list[i][0] - emb_list[i+1][0]))
                if len(dists) >= 2:
                    cv = np.std(dists) / (np.mean(dists) + 1e-8)
                    coherence_vals.append(1.0 / (1.0 + cv))

        # RANO: detect >= 25% growth events
        for i in range(len(seqs) - 1):
            v1 = fast_vol_lookup(pid, seqs[i][0])
            v2 = fast_vol_lookup(pid, seqs[-1][0])
            if v1 and v2 and v1 > 1.0:
                grown = 1 if (v2 - v1) / v1 >= 0.25 else 0
                rano_pairs.append((pid, grown))
                break

    # -- Compute T1: Spearman --
    if len(spearman_pairs) >= 10:
        x, y = zip(*spearman_pairs)
        rho, p = spearmanr(x, y)
        metrics["T1_spearman_wt"] = rho
        print(f"  {label} T1_spearman_wt: {rho:.3f} (p={p:.4f})")
    else:
        metrics["T1_spearman_wt"] = 0.0
        print(f"  {label} T1_spearman_wt: 0.000 (not enough pairs)")

    # -- Compute T8: Kendall tau (trajectory-level) --
    if is_trajectory and len(kendall_pairs) >= 10:
        emb_arr = np.stack([p[0] for p in kendall_pairs])
        deltas = np.array([p[1] for p in kendall_pairs])
        # Use first PC of trajectory embeddings
        pca = PCA(n_components=1)
        pc1 = pca.fit_transform(emb_arr).squeeze()
        tau, p = kendalltau(pc1, deltas)
        metrics["T8_kendall_tau"] = abs(tau)
        print(f"  {label} T8_kendall_tau: {abs(tau):.3f} (p={p:.4f})")

        # Also compute Spearman on PC1
        rho_pc1, _ = spearmanr(pc1, deltas)
        metrics["T1_spearman_pc1"] = abs(rho_pc1)
        print(f"  {label} T1_spearman_pc1: {abs(rho_pc1):.3f}")
    elif not is_trajectory and len(spearman_pairs) >= 10:
        x, y = zip(*spearman_pairs)
        tau, p = kendalltau(x, y)
        metrics["T8_kendall_tau"] = abs(tau)
        print(f"  {label} T8_kendall_tau: {abs(tau):.3f}")

    # -- T5: Coherence --
    if coherence_vals:
        metrics["T5_coherence"] = np.mean(coherence_vals)
        print(f"  {label} T5_coherence: {np.mean(coherence_vals):.3f}")

    # -- T4: RANO AUC --
    if len(rano_pairs) >= 10:
        rano_pids_used = [p[0] for p in rano_pairs]
        rano_labels = np.array([p[1] for p in rano_pairs])
        if is_trajectory and rano_labels.sum() > 0 and rano_labels.sum() < len(rano_labels):
            rano_embs = np.stack([patient_embs[pid] for pid in rano_pids_used])
            X = StandardScaler().fit_transform(rano_embs)
            try:
                cv = StratifiedKFold(n_splits=min(5, int(rano_labels.sum())), shuffle=True, random_state=42)
                scores = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                                        X, rano_labels, cv=cv, scoring="roc_auc")
                metrics["T4_rano_auc"] = np.mean(scores)
                print(f"  {label} T4_rano_auc: {np.mean(scores):.3f}")
            except Exception as e:
                metrics["T4_rano_auc"] = 0.5
                print(f"  {label} T4_rano_auc: 0.500 (error: {e})")

    return metrics


# ============================================================
# RUN TEMPORAL TESTS: TaViT
# ============================================================
print("\n-- TaViT Trajectory Embeddings --")
tavit_embs = {pid: traj_npz[pid].astype(np.float32) for pid in traj_pids}
results["tavit"] = compute_temporal_metrics(
    tavit_embs, static_patient_seqs, label="TaViT")

# ============================================================
# RUN TEMPORAL TESTS: Static Hybrid (for comparison)
# ============================================================
if static_npz is not None:
    print("\n-- Static Hybrid Embeddings --")
    static_embs = {}
    for pid in traj_pids:
        if pid in static_patient_seqs:
            seqs = static_patient_seqs[pid]
            static_embs[pid] = [(s[1], s[0]) for s in seqs]

    results["static_hybrid"] = compute_temporal_metrics(
        static_embs, static_patient_seqs, label="Static")


In [ ]:
# ============================================================
# NEW TRAJECTORY-SPECIFIC TESTS
# ============================================================
print("\n" + "=" * 60)
print("  NEW TRAJECTORY-SPECIFIC TESTS")
print("=" * 60)

# Only patients with volume labels
labeled_pids = [pid for pid in traj_pids if pid in patient_labels]
if len(labeled_pids) < 20:
    print(f"  Not enough labeled patients ({len(labeled_pids)})")
else:
    X_traj = np.stack([traj_npz[pid].astype(np.float32) for pid in labeled_pids])
    y_delta = np.array([patient_labels[pid]["delta_vol"] for pid in labeled_pids])
    y_response = np.array([patient_labels[pid]["response"] for pid in labeled_pids])
    X_scaled = StandardScaler().fit_transform(X_traj)

    # ---- NEW T9: Progression Regression R2 ----
    print(f"\n  N9: Progression Regression (n={len(labeled_pids)})")
    try:
        rf_reg = RandomForestRegressor(n_estimators=200, random_state=42)
        r2_scores = cross_val_score(rf_reg, X_scaled, y_delta, cv=5, scoring="r2")
        ridge_reg = Ridge(alpha=1.0)
        r2_ridge = cross_val_score(ridge_reg, X_scaled, y_delta, cv=5, scoring="r2")
        results["tavit"]["N9_progression_R2_rf"] = np.mean(r2_scores)
        results["tavit"]["N9_progression_R2_ridge"] = np.mean(r2_ridge)
        print(f"    RF R2:    {np.mean(r2_scores):.3f} +/- {np.std(r2_scores):.3f}")
        print(f"    Ridge R2: {np.mean(r2_ridge):.3f} +/- {np.std(r2_ridge):.3f}")
    except Exception as e:
        print(f"    Error: {e}")

    # ---- NEW T10: Response Classification (3-class) ----
    print(f"\n  N10: Response Classification (progressive/stable/responder)")
    resp_counts = defaultdict(int)
    for r in y_response:
        resp_counts[r] += 1
    print(f"    Distribution: {dict(resp_counts)}")

    # Need at least 2 classes with >= 3 samples
    valid_classes = [c for c, n in resp_counts.items() if n >= 3]
    if len(valid_classes) >= 2:
        mask = np.isin(y_response, valid_classes)
        X_filt = X_scaled[mask]
        y_filt = y_response[mask]
        try:
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            rf_cls = RandomForestClassifier(n_estimators=200, random_state=42)
            f1_scores = cross_val_score(rf_cls, X_filt, y_filt, cv=cv,
                                        scoring="f1_weighted")
            results["tavit"]["N10_response_F1"] = np.mean(f1_scores)
            print(f"    RF Weighted F1: {np.mean(f1_scores):.3f} +/- {np.std(f1_scores):.3f}")

            # Also compute per-class report on full data
            rf_cls.fit(X_filt, y_filt)
            y_pred = rf_cls.predict(X_filt)
            print(f"\n    Full-data classification report:")
            print(classification_report(y_filt, y_pred, zero_division=0))
        except Exception as e:
            print(f"    Error: {e}")
    else:
        print(f"    Not enough classes with >=3 samples: {dict(resp_counts)}")

    # ---- NEW T11: Trajectory Clustering Quality ----
    print(f"\n  N11: Trajectory Clustering Quality")
    try:
        for k in [2, 3]:
            km = KMeans(n_clusters=k, random_state=42, n_init=10)
            clust_labels = km.fit_predict(X_scaled)
            sil = silhouette_score(X_scaled, clust_labels)
            results["tavit"][f"N11_silhouette_k{k}"] = sil
            print(f"    K={k}: Silhouette = {sil:.3f}")

            # Check if clusters align with response categories
            if len(valid_classes) >= 2:
                from sklearn.metrics import adjusted_rand_score
                # Map response to int for ARI
                resp_map = {c: i for i, c in enumerate(sorted(valid_classes))}
                y_int = np.array([resp_map.get(r, -1) for r in y_response])
                valid_ari = y_int >= 0
                if valid_ari.sum() >= 10:
                    ari = adjusted_rand_score(y_int[valid_ari], clust_labels[valid_ari])
                    results["tavit"][f"N11_ari_k{k}"] = ari
                    print(f"    K={k}: ARI with response labels = {ari:.3f}")
    except Exception as e:
        print(f"    Error: {e}")

    # ---- NEW T12: Treatment Separation (Cohen's d) ----
    print(f"\n  N12: Progressive vs Stable Separation")
    prog_idx = y_response == "progressive"
    stab_idx = y_response == "stable"
    if prog_idx.sum() >= 5 and stab_idx.sum() >= 5:
        prog_embs = X_traj[prog_idx]
        stab_embs = X_traj[stab_idx]
        mean_diff = np.linalg.norm(prog_embs.mean(0) - stab_embs.mean(0))
        pooled_std = np.sqrt((prog_embs.var(0) + stab_embs.var(0)).mean())
        cohens_d = mean_diff / (pooled_std + 1e-8)
        results["tavit"]["N12_cohens_d"] = cohens_d
        print(f"    Cohen's d (progressive vs stable) = {cohens_d:.3f}")
        print(f"    {'GOOD' if cohens_d > 0.5 else 'WEAK'} separation (threshold: 0.5)")

        # Mann-Whitney U test on trajectory norms
        prog_norms = np.linalg.norm(prog_embs, axis=1)
        stab_norms = np.linalg.norm(stab_embs, axis=1)
        u_stat, u_p = mannwhitneyu(prog_norms, stab_norms, alternative="two-sided")
        results["tavit"]["N12_mannwhitney_p"] = u_p
        print(f"    Mann-Whitney U p-value = {u_p:.4f}")
    else:
        print(f"    Not enough samples (prog={prog_idx.sum()}, stab={stab_idx.sum()})")

    # ---- NEW T13: Velocity Prediction ----
    print(f"\n  N13: Velocity Prediction (volume change rate)")
    n_scans = np.array([patient_labels[pid].get("n_scans", 2) for pid in labeled_pids])
    # Velocity = delta_vol / n_scans (proxy for rate)
    velocity = y_delta / (n_scans - 1 + 1e-8)
    try:
        rf_vel = RandomForestRegressor(n_estimators=200, random_state=42)
        vel_r2 = cross_val_score(rf_vel, X_scaled, velocity, cv=5, scoring="r2")
        results["tavit"]["N13_velocity_R2"] = np.mean(vel_r2)
        print(f"    Velocity R2 (RF): {np.mean(vel_r2):.3f} +/- {np.std(vel_r2):.3f}")
    except Exception as e:
        print(f"    Error: {e}")


In [ ]:
# ============================================================
# HEAD-TO-HEAD: TaViT vs STATIC vs CNN BASELINE
# ============================================================
print("\n" + "=" * 60)
print("  HEAD-TO-HEAD COMPARISON")
print("=" * 60)

# Load B1 cached results if available
b1_path = find_file(["unified_eval_results.json"])
b1_results = {}
if b1_path:
    with open(b1_path) as f:
        b1_results = _json.load(f)
    print(f"  Loaded B1 results: {list(b1_results.keys())}")

# Build comparison table for temporal metrics
temporal_metrics = ["T1_spearman_wt", "T8_kendall_tau", "T4_rano_auc",
                    "T5_coherence"]

print(f"\n  {'Metric':<30} {'CNN':>8} {'ViT-Base':>10} {'ViT-Hybrid':>12} {'TaViT':>8}  Note")
print(f"  {'-'*85}")

for metric in temporal_metrics:
    cnn_val = b1_results.get("nnunet", {}).get(metric, None)
    base_val = b1_results.get("swinunetr_base", {}).get(metric, None)
    hybrid_val = b1_results.get("swinunetr_hybrid",
                  b1_results.get("swinunetr_global", {})).get(metric, None)
    tavit_val = results.get("tavit", {}).get(metric, None)

    # Special: for TaViT T1, prefer pc1 version if available
    if metric == "T1_spearman_wt" and "T1_spearman_pc1" in results.get("tavit", {}):
        tavit_val_alt = results["tavit"]["T1_spearman_pc1"]
        if tavit_val_alt is not None and (tavit_val is None or abs(tavit_val_alt) > abs(tavit_val)):
            tavit_val = tavit_val_alt

    def fmt(v):
        return f"{v:.3f}" if v is not None else "  --  "

    # Find best
    vals = {"CNN": cnn_val, "ViT-Base": base_val, "ViT-Hybrid": hybrid_val, "TaViT": tavit_val}
    valid = {k: v for k, v in vals.items() if v is not None}
    if valid:
        best_name = max(valid, key=lambda k: abs(valid[k]))
        note = f"<- {best_name}"
    else:
        note = ""

    print(f"  {metric:<30} {fmt(cnn_val):>8} {fmt(base_val):>10} "
          f"{fmt(hybrid_val):>12} {fmt(tavit_val):>8}  {note}")

# New metrics (TaViT only)
print(f"\n  {'-- TaViT-ONLY Metrics --':<30}")
tavit_only = ["N9_progression_R2_rf", "N9_progression_R2_ridge",
              "N10_response_F1", "N11_silhouette_k2", "N11_silhouette_k3",
              "N11_ari_k3", "N12_cohens_d", "N13_velocity_R2"]
for metric in tavit_only:
    val = results.get("tavit", {}).get(metric, None)
    if val is not None:
        print(f"  {metric:<30} {val:>8.3f}")

# ============================================================
# PASS/FAIL SUMMARY
# ============================================================
print(f"\n{'='*60}")
print(f"  TaViT PASS/FAIL SUMMARY")
print(f"{'='*60}")

thresholds = {
    "T1_spearman_wt": 0.3,
    "T1_spearman_pc1": 0.3,
    "T8_kendall_tau": 0.3,
    "T4_rano_auc": 0.65,
    "N9_progression_R2_rf": 0.2,
    "N10_response_F1": 0.5,
    "N12_cohens_d": 0.5,
    "N13_velocity_R2": 0.1,
}

n_pass = 0; n_fail = 0; n_total = 0
for metric, thresh in thresholds.items():
    val = results.get("tavit", {}).get(metric, None)
    if val is None:
        continue
    n_total += 1
    passed = abs(val) >= thresh
    if passed:
        n_pass += 1
    else:
        n_fail += 1
    status = "PASS" if passed else "FAIL"
    print(f"  {metric:<30} {val:>8.3f}  thresh={thresh:<6}  {status}")

print(f"\n  Score: {n_pass}/{n_total} PASS | {n_fail}/{n_total} FAIL")


In [ ]:
# ============================================================
# VISUALIZATIONS
# ============================================================
print("\n" + "=" * 60)
print("  TRAJECTORY VISUALIZATIONS")
print("=" * 60)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.manifold import TSNE
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("  matplotlib not available -- skipping plots")

if HAS_MPL and len(patient_labels) >= 20:
    labeled_pids_vis = [pid for pid in traj_pids if pid in patient_labels]
    X_vis = np.stack([traj_npz[pid].astype(np.float32) for pid in labeled_pids_vis])
    y_resp = np.array([patient_labels[pid]["response"] for pid in labeled_pids_vis])
    y_delta = np.array([patient_labels[pid]["delta_vol"] for pid in labeled_pids_vis])

    # ---- t-SNE colored by response ----
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X_vis)-1))
    X_2d = tsne.fit_transform(X_vis)

    color_map = {"progressive": "#e74c3c", "stable": "#3498db", "responder": "#2ecc71"}
    for resp, color in color_map.items():
        mask = y_resp == resp
        if mask.sum() > 0:
            axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=resp,
                          alpha=0.6, s=30, edgecolors="white", linewidths=0.3)
    axes[0].set_title("TaViT Trajectories -- by Response", fontsize=13)
    axes[0].legend()
    axes[0].set_xlabel("t-SNE 1")
    axes[0].set_ylabel("t-SNE 2")

    # ---- t-SNE colored by volume change (continuous) ----
    sc = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=np.clip(y_delta, -2, 2),
                        cmap="RdYlGn_r", alpha=0.6, s=30, edgecolors="white",
                        linewidths=0.3)
    plt.colorbar(sc, ax=axes[1], label="Volume Change (fraction)")
    axes[1].set_title("TaViT Trajectories -- by Volume Change", fontsize=13)
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")

    plt.tight_layout()
    plt.savefig(FIG_DIR / "tavit_tsne_response.png", dpi=150, bbox_inches="tight")
    print(f"  Saved: {FIG_DIR / 'tavit_tsne_response.png'}")
    plt.close()

    # ---- PCA: PC1 vs volume change scatter ----
    fig, ax = plt.subplots(figsize=(8, 6))
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_vis)
    sc = ax.scatter(X_pca[:, 0], y_delta, c=[color_map.get(r, "gray") for r in y_resp],
                   alpha=0.5, s=20)
    ax.set_xlabel(f"PC1 (explained var: {pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel("Volume Change (fraction)")
    ax.set_title("TaViT PC1 vs Volume Change")
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
    ax.axhline(y=0.25, color="red", linestyle="--", alpha=0.3, label="RANO +25%")
    ax.axhline(y=-0.25, color="green", linestyle="--", alpha=0.3, label="RANO -25%")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "tavit_pc1_vs_volume.png", dpi=150, bbox_inches="tight")
    print(f"  Saved: {FIG_DIR / 'tavit_pc1_vs_volume.png'}")
    plt.close()

    # ---- Embedding norm distribution by response ----
    fig, ax = plt.subplots(figsize=(8, 5))
    for resp, color in color_map.items():
        mask = y_resp == resp
        if mask.sum() > 0:
            norms = np.linalg.norm(X_vis[mask], axis=1)
            ax.hist(norms, bins=20, alpha=0.5, color=color, label=resp)
    ax.set_xlabel("Trajectory Embedding Norm")
    ax.set_ylabel("Count")
    ax.set_title("Embedding Norm Distribution by Response")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "tavit_norm_by_response.png", dpi=150, bbox_inches="tight")
    print(f"  Saved: {FIG_DIR / 'tavit_norm_by_response.png'}")
    plt.close()

    print(f"  All figures in: {FIG_DIR}")


In [ ]:
# ============================================================
# SAVE RESULTS + FINAL SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)

# Save results JSON
with open(OUTPUT_ROOT / "tavit_eval_results.json", "w") as f:
    _json.dump(results, f, indent=2, default=str)
print(f"  Saved: {OUTPUT_ROOT / 'tavit_eval_results.json'}")

# ============================================================
# IMPROVEMENT REPORT
# ============================================================
print(f"\n{'='*60}")
print(f"  TaViT vs STATIC HYBRID -- IMPROVEMENT REPORT")
print(f"{'='*60}")

improvements = {
    "T1_spearman_wt": {"static": 0.094, "description": "Temporal correlation"},
    "T8_kendall_tau": {"static": 0.354, "description": "Trajectory ordering"},
    "T4_rano_auc":    {"static": 0.603, "description": "RANO detection"},
}

for metric, info in improvements.items():
    static_val = info["static"]
    tavit_val = results.get("tavit", {}).get(metric, None)

    # For T1, also check pc1 version
    if metric == "T1_spearman_wt":
        alt = results.get("tavit", {}).get("T1_spearman_pc1", None)
        if alt is not None and (tavit_val is None or abs(alt) > abs(tavit_val)):
            tavit_val = alt

    if tavit_val is not None:
        delta = abs(tavit_val) - abs(static_val)
        pct = 100 * delta / (abs(static_val) + 1e-8)
        status = "IMPROVED" if delta > 0 else "same/worse"
        print(f"  {info['description']:<25} static={abs(static_val):.3f} -> "
              f"TaViT={abs(tavit_val):.3f}  ({pct:+.0f}%)  {status}")

print(f"\n  All outputs: {OUTPUT_ROOT}")
print(f"\n  Phase 4A complete.")
print(f"  Next: Use trajectory embeddings in Phase 4 LLM narrative generation")
